# Module 3 — Measuring Relationships: Correlation (and its traps)

"EPV varies by day of week." Maybe — but **correlation is a hypothesis, not a finding.**
Two traps bite NI analysts constantly:
1. **Outliers distort Pearson** correlation → rank-based (Spearman/Kendall) is safer on EPV.
2. **A confound fakes a relationship** → the weekend EPV "dip" is really a *channel-mix* effect.

Runs on the real NI `online_banking` data.

In [ ]:
import sys
from pathlib import Path
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / "src" / "ni_style.py").exists():
        sys.path.insert(0, str(_c / "src"))
        sys.path.insert(0, str(_c / ".claude" / "skills" / "_lib"))
        break
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import ni_style as ni      # chart style + palette
import ni_core as C        # real NI data + primitives
ni.set_style()

visits = C.load_visits()
print(f"Loaded {len(visits):,} visits | {visits['date'].min().date()} -> {visits['date'].max().date()}")
visits.head()

In [ ]:
# Daily aggregates we'll relate to each other
daily = visits.groupby("date", observed=True).agg(
    EPV=("revenue","mean"), conv=("converted","mean"), visits=("visit_iid","size")).reset_index()
daily["dow"] = pd.Categorical(daily["date"].dt.strftime("%a"), categories=ni.DOW_ORDER, ordered=True)
daily.head()

## 1. The raw picture: EPV by day of week

There *is* a weekly pattern — weekends look a little poorer than weekdays. A small dip…
but is it a *day* effect, or is something else riding along with the weekend?

In [ ]:
by_dow = visits.groupby("day_of_week", observed=True)["revenue"].mean().reindex(ni.DOW_ORDER)
fig, ax = plt.subplots(figsize=(11, 5))
colors = [ni.BLUE if d not in ("Sat","Sun") else ni.ORANGE for d in by_dow.index]
ax.bar(by_dow.index, by_dow.values, color=colors)
for i, v in enumerate(by_dow.values):
    ax.text(i, v+0.02, f"${v:.2f}", ha="center", fontweight="bold", color=ni.NAVY)
ax.set_ylabel("EPV ($)")
ni.titlebox(ax, "EPV by day of week — a small weekend dip",
            "orange = weekend.  Looks like a 'day' effect… is it real?")
fig.tight_layout(); ni.savefig(fig, "m3_epv_by_dow"); plt.show()

## 2. Three correlation coefficients — and why Pearson can mislead

**Pearson** measures *linear* association and is sensitive to outliers.
**Spearman** and **Kendall** work on *ranks* — robust to skew and whales.
We relate **daily conversion rate to daily EPV**: they genuinely move together, but a
few whale days distort the linear fit. On NI metrics, **default to Spearman.**

In [ ]:
# Relate daily conversion rate to daily EPV — whale days make Pearson understate it
x, y = daily["conv"].values, daily["EPV"].values
pear = stats.pearsonr(x, y); spear = stats.spearmanr(x, y); kend = stats.kendalltau(x, y)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(x*100, y, s=42, color=ni.BLUE, alpha=0.7, edgecolor="white")
hi = y > np.percentile(y, 95)                       # whale-driven outlier days
ax[0].scatter(x[hi]*100, y[hi], s=90, facecolor="none", edgecolor=ni.RED, linewidth=2, label="whale days (top-5% EPV)")
ax[0].set_xlabel("daily conversion rate (%)"); ax[0].set_ylabel("daily EPV ($)"); ax[0].legend()
ni.titlebox(ax[0], "Daily EPV vs conversion rate",
            f"Pearson r={pear.statistic:.2f}  |  Spearman rho={spear.statistic:.2f}")

methods = ["Pearson\n(linear)", "Spearman\n(rank)", "Kendall\n(rank)"]
vals = [pear.statistic, spear.statistic, kend.statistic]
ax[1].bar(methods, vals, color=[ni.RED, ni.GREEN, ni.TEAL])
for i, v in enumerate(vals):
    ax[1].text(i, v+0.01, f"{v:.2f}", ha="center", fontweight="bold", color=ni.NAVY)
ax[1].set_ylabel("correlation coefficient"); ax[1].set_ylim(0, 0.6)
ni.titlebox(ax[1], "Same data, three coefficients", "Pearson sits below the rank measures — whales pulling it down")
fig.tight_layout(); ni.savefig(fig, "m3_three_correlations"); plt.show()

### Outliers really do move Pearson

Remove just the top-5% whale days and watch **Pearson jump while Spearman barely
budges** — proof that the rank measure is telling you the stable story.

In [ ]:
keep = ~hi
p_all, p_trim = stats.pearsonr(x, y).statistic, stats.pearsonr(x[keep], y[keep]).statistic
s_all, s_trim = stats.spearmanr(x, y).statistic, stats.spearmanr(x[keep], y[keep]).statistic

fig, ax = plt.subplots(figsize=(9, 5))
xb = np.arange(2); w = 0.35
ax.bar(xb-w/2, [p_all, s_all], w, label="all days", color=ni.NAVY)
ax.bar(xb+w/2, [p_trim, s_trim], w, label="whale days removed", color=ni.SKY)
ax.set_xticks(xb); ax.set_xticklabels(["Pearson", "Spearman"]); ax.set_ylabel("coefficient"); ax.legend()
for i,(a,b) in enumerate([(p_all,p_trim),(s_all,s_trim)]):
    ax.text(i-w/2, a+0.01, f"{a:.2f}", ha="center", fontweight="bold")
    ax.text(i+w/2, b+0.01, f"{b:.2f}", ha="center", fontweight="bold")
ni.titlebox(ax, "Pearson is fragile, Spearman is stable",
            "removing 5% of days swings Pearson far more than Spearman")
fig.tight_layout(); ni.savefig(fig, "m3_pearson_fragile"); plt.show()

## 3. The confound: it was never about the *day*

The weekend EPV dip is real in the raw average — but it's driven by **what kind of
traffic runs on weekends**, not the day itself. Weekends carry far more low-EPV
**Social** traffic (≈31% of weekend visits vs ≈10% on weekdays), and Social EPV is
about half the site average. **Control for channel and the dip flips sign** — within
*every* channel, weekend EPV is flat or *higher*. A textbook Simpson's reversal.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5.2))

# Left: the channel MIX weekday vs weekend — the real driver
mix = (visits.assign(part=np.where(visits.is_weekend,"Weekend","Weekday"))
       .pivot_table(index="part", columns="channel", values="visit_iid", aggfunc="size"))
mix = mix.div(mix.sum(axis=1), axis=0)[["Google","Bing","Organic","Social"]]
bottom = np.zeros(len(mix))
for i, v in enumerate(mix.columns):
    ax[0].bar(mix.index, mix[v], bottom=bottom, label=v, color=ni.SEQ[i % len(ni.SEQ)])
    bottom += mix[v].values
ax[0].set_ylabel("share of traffic"); ax[0].legend(ncol=2, fontsize=9, loc="lower center")
ni.titlebox(ax[0], "WHY: the channel mix differs by day",
            "weekends carry far more low-EPV Social traffic")

# Right: EPV weekend vs weekday, overall vs WITHIN one channel (Social)
parts = ["Weekday","Weekend"]
overall = [visits.loc[~visits.is_weekend,"revenue"].mean(), visits.loc[visits.is_weekend,"revenue"].mean()]
soc = visits[visits.channel=="Social"]
within = [soc.loc[~soc.is_weekend,"revenue"].mean(), soc.loc[soc.is_weekend,"revenue"].mean()]
xb = np.arange(2); w = 0.36
ax[1].bar(xb-w/2, overall, w, label="overall EPV", color=ni.BLUE)
ax[1].bar(xb+w/2, within, w, label="within 'Social' only", color=ni.GREEN)
ax[1].set_xticks(xb); ax[1].set_xticklabels(parts); ax[1].set_ylabel("EPV ($)"); ax[1].legend()
ni.titlebox(ax[1], "The 'weekend dip' flips sign once mix is held fixed",
            f"overall {overall[1]/overall[0]-1:+.0%}  ->  within-Social {within[1]/within[0]-1:+.0%}")
fig.tight_layout(); ni.savefig(fig, "m3_confound"); plt.show()

### ✅ Takeaway

> **On NI data, default to Spearman, and treat every correlation as a *question*.**
> "EPV drops on weekends" was a **mix artifact** — the day didn't cause anything; the
> *channel composition* of weekend traffic did (far more low-EPV Social). Before acting
> on any correlation, ask: *what third variable changes alongside both of these?*

**Next:** the Slack message claims Google ≠ Bing on conversion. Is that gap real,
or could it be luck? → *Module 4.*